# U5_06 — Graph RAG y Memoria en Grafo

**Unidad:** 5 — Sistemas Multi-Agente Modernos  
**Dificultad:** Avanzada ★★★★☆  
**Duración estimada:** 5-7 horas  
**Prereq. mínimo:** U5_05 (RAG y Memoria Avanzada)  
**Entorno:** `ia_multiagente` (Python 3.11)

---

## Objetivos de Aprendizaje

Al completar esta notebook el estudiante podrá:
1. Diferenciar cuándo una base de datos en grafo supera a un vector store
2. Construir un knowledge graph incremental con Kuzu (embebido) y NetworkX
3. Implementar consultas Cypher como tools de un agente LLM
4. Crear un sistema de Graph RAG híbrido (vector + grafo)
5. Gestionar memoria episódica estructurada con Graphiti + Neo4j
6. Resolver consultas multi-hop que los vector stores no pueden responder

---

## Tabla de Contenidos

1. [¿Por qué grafos? Limitaciones de los vector stores](#por-que-grafos)
2. [Warm-Up: Entorno y Datos](#warmup)
3. [Nivel 1: NetworkX — Grafos en Memoria](#networkx)
4. [Nivel 2: Kuzu — Grafo Embebido Persistente](#kuzu)
5. [Nivel 3: Neo4j — Grafo con Servidor](#neo4j)
6. [Cypher como Tool: Agente que Consulta el Grafo](#cypher-tool)
7. [Graph RAG: Combinar Vector Store y Knowledge Graph](#graph-rag)
8. [Graphiti: Memoria Episódica en Grafo](#graphiti)
9. [Benchmark: Multi-hop Queries — Grafo vs Vector](#benchmark)
10. [Ejercicio: Construir tu Propio Knowledge Graph](#ejercicio)

---

> **Criterios de evaluación:** Demostrar ≥1 consulta multi-hop que falle en el vector store pero que el grafo resuelva correctamente; implementar Graph RAG híbrido funcional; memoria episódica con Graphiti guardando y recuperando episodios.
>
> **Errores comunes:** Usar un grafo donde un vector store es suficiente (over-engineering); escribir Cypher sin sanitizar inputs del usuario (inyección); confundir grafos de conocimiento con grafos computacionales (PyTorch/JAX).

## 1. ¿Por Qué Grafos? Limitaciones de los Vector Stores <a id='por-que-grafos'></a>

### El problema central: las relaciones

Un vector store almacena textos como vectores en un espacio n-dimensional. Es excelente para preguntas del tipo:

> *"¿Qué papers hablan sobre transformers?"* → recuperación por similitud semántica

Pero falla ante preguntas que requieren **razonar sobre relaciones**:

> *"¿Qué investigadores han colaborado con colegas que trabajan en el mismo lab que el autor de este paper?"*

Esta es una consulta **multi-hop**: requiere seguir relaciones (autor → paper → lab → colegas → colaboraciones). Los vector stores no codifican relaciones — solo codifican contenido.

### Cuándo usar cada tecnología

| Pregunta | Tecnología óptima | Razón |
|----------|-------------------|-------|
| Similitud semántica entre textos | Vector store (ChromaDB, Pinecone) | Embeddings capturan significado |
| Relaciones entre entidades | Knowledge graph (Neo4j, Kuzu) | Estructura nativa de grafos |
| Caminos entre nodos | Knowledge graph + Cypher | Path queries nativas |
| Contexto de conversación | Graphiti / episodic memory | Episodios en grafo temporal |
| "¿Qué es similar a X?" | Vector store | Búsqueda por vecinos |
| "¿Quién conoce a quién a través de quién?" | Knowledge graph | Multi-hop queries |
| Las dos preguntas en el mismo sistema | Graph RAG híbrido | Combinar ambas |

### Taxonomía de grafos que usaremos

```
NetworkX  →  simple, en memoria, sin persistencia, ideal para prototipos
    ↓
Kuzu      →  embebido (como SQLite), persistente, sin servidor, Cypher nativo
    ↓
Neo4j     →  servidor, producción, ACID, Cypher avanzado, visualización
    ↓
GraphRAG  →  capa sobre el grafo orientada a LLMs
    ↓
Graphiti  →  memoria episódica en grafo para agentes multi-turno
```

## 2. Warm-Up: Entorno y Datos <a id='warmup'></a>

In [ ]:
# ============================================================
# WARM-UP: U5_08 — Graph RAG y Memoria en Grafo
# ============================================================
import subprocess, sys, os
from dotenv import load_dotenv

load_dotenv()

# Verificar paquetes
REQUIRED = ["networkx", "langchain_google_genai", "chromadb"]
OPTIONAL = [
    ("kuzu", "Kuzu: grafo embebido — pip install kuzu"),
    ("neo4j", "Neo4j driver — pip install neo4j (requiere servidor)"),
    ("graphiti_core", "Graphiti — pip install graphiti-core (requiere Neo4j)"),
]

# Instalar langchain-openai si no está disponible (necesario para OpenRouter)
try:
    __import__("langchain_openai")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain-openai==0.3.12", "-q"])

print("--- Paquetes requeridos ---")
for pkg in REQUIRED:
    try:
        __import__(pkg)
        print(f"  OK: {pkg}")
    except ImportError:
        print(f"  FALTA (instalar): {pkg}")

print("\n--- Paquetes opcionales ---")
AVAILABLE = {}
for pkg, note in OPTIONAL:
    try:
        __import__(pkg)
        AVAILABLE[pkg] = True
        print(f"  OK: {pkg}")
    except ImportError:
        AVAILABLE[pkg] = False
        print(f"  No disponible: {note}")

print("\n--- Variables de entorno ---")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
GOOGLE_API_KEY     = os.environ.get("GOOGLE_API_KEY")
OPENAI_API_KEY     = os.environ.get("OPENAI_API_KEY")

for var in ["OPENROUTER_API_KEY", "GOOGLE_API_KEY", "OPENAI_API_KEY", "NEO4J_URI", "NEO4J_USER", "NEO4J_PASSWORD"]:
    val = os.environ.get(var)
    print(f"  {'OK' if val else 'no configurada'}: {var}")

backend = (
    f"OpenRouter ({os.environ.get('OPENROUTER_MODEL', 'llama-3.2-3b-instruct:free')})" if OPENROUTER_API_KEY else
    "Gemini 2.0 Flash" if GOOGLE_API_KEY else
    "GPT-4o-mini" if OPENAI_API_KEY else
    "Ollama/llama3.2 (local)"
)

print(f"\nBackend LLM: {backend}")
print("\nWarm-up completado.")
print(f"Kuzu disponible: {AVAILABLE.get('kuzu', False)}")
print(f"Neo4j disponible: {AVAILABLE.get('neo4j', False)}")
print(f"Graphiti disponible: {AVAILABLE.get('graphiti_core', False)}")


--- Paquetes requeridos ---
  OK: networkx
  OK: langchain_google_genai
  OK: chromadb

--- Paquetes opcionales ---
  OK: kuzu
  OK: neo4j
  No disponible: Graphiti — pip install graphiti-core (requiere Neo4j)

--- Variables de entorno ---
  OK: OPENROUTER_API_KEY
  OK: GOOGLE_API_KEY
  no configurada: OPENAI_API_KEY
  OK: NEO4J_URI
  no configurada: NEO4J_USER
  OK: NEO4J_PASSWORD

Backend LLM: OpenRouter (google/gemini-2.5-flash)

Warm-up completado.
Kuzu disponible: True
Neo4j disponible: True
Graphiti disponible: False


In [1]:
# Dataset de ejemplo: comunidad de investigación en nanotecnología
# Este dataset representa el dominio del curso IA Nanotecnología

RESEARCH_KG_DATA = {
    "researchers": [
        {"id": "r1", "name": "Ana García",    "lab": "NanoAI Lab",    "institution": "UNAM"},
        {"id": "r2", "name": "Carlos Ruiz",   "lab": "NanoAI Lab",    "institution": "UNAM"},
        {"id": "r3", "name": "María López",   "lab": "Quantum Lab",   "institution": "IPN"},
        {"id": "r4", "name": "Pedro Martínez","lab": "Quantum Lab",   "institution": "IPN"},
        {"id": "r5", "name": "Laura Chen",    "lab": "ML for Nano",   "institution": "CINVESTAV"},
        {"id": "r6", "name": "Diego Torres",  "lab": "ML for Nano",   "institution": "CINVESTAV"},
    ],
    "papers": [
        {"id": "p1", "title": "Gold nanoparticles for drug delivery",    "year": 2024, "field": "nanomedicine"},
        {"id": "p2", "title": "Quantum sensing with NV centers",          "year": 2024, "field": "quantum_sensing"},
        {"id": "p3", "title": "ML for nanoparticle synthesis prediction", "year": 2025, "field": "ml_nano"},
        {"id": "p4", "title": "Multimodal LLMs for materials science",    "year": 2025, "field": "ml_nano"},
        {"id": "p5", "title": "Quantum-classical hybrid algorithms",       "year": 2025, "field": "quantum_ml"},
    ],
    "authorships": [  # (researcher_id, paper_id)
        ("r1", "p1"), ("r2", "p1"), ("r3", "p2"),
        ("r4", "p2"), ("r5", "p3"), ("r6", "p3"),
        ("r1", "p4"), ("r5", "p4"),  # colaboración cross-lab!
        ("r3", "p5"), ("r5", "p5"),  # otra colaboración cross-lab!
    ],
    "citations": [  # (paper_id_citante, paper_id_citado)
        ("p3", "p1"), ("p4", "p1"), ("p4", "p3"), ("p5", "p2"), ("p5", "p3"),
    ],
}

print("Dataset cargado:")
print(f"  Investigadores: {len(RESEARCH_KG_DATA['researchers'])}")
print(f"  Papers: {len(RESEARCH_KG_DATA['papers'])}")
print(f"  Autorías: {len(RESEARCH_KG_DATA['authorships'])}")
print(f"  Citaciones: {len(RESEARCH_KG_DATA['citations'])}")
print()
print("Pregunta multi-hop que plantearemos:")
print("  '¿Qué investigadores de labs distintos han colaborado en papers sobre ML para nano?'")
print("  (Requiere seguir: RESEARCHER → PAPER ← RESEARCHER, filtrando por lab distinto y campo)")

Dataset cargado:
  Investigadores: 6
  Papers: 5
  Autorías: 10
  Citaciones: 5

Pregunta multi-hop que plantearemos:
  '¿Qué investigadores de labs distintos han colaborado en papers sobre ML para nano?'
  (Requiere seguir: RESEARCHER → PAPER ← RESEARCHER, filtrando por lab distinto y campo)


## 3. Nivel 1: NetworkX — Grafos en Memoria <a id='networkx'></a>

NetworkX es el punto de entrada ideal: cero dependencias de servidor, funciona en cualquier máquina, y permite explorar la API de grafos antes de escalar.

**Cuándo usarlo:**
- Prototipos y notebooks educativos
- Grafos pequeños (<100K nodos) que caben en RAM
- Análisis de grafos sin necesidad de persistencia entre sesiones

**Cuándo NO usarlo:**
- Grafos que crecen con el tiempo (usar Kuzu o Neo4j)
- Consultas concurrentes desde múltiples procesos
- Grafos de más de 1M de nodos (el rendimiento cae)

**Conexión pedagógica:** Los algoritmos de grafo que veremos (shortest path, community detection, PageRank) son los mismos en los tres niveles — solo cambia el backend de almacenamiento. Eso hace que el conocimiento sea transferible.


In [2]:
import networkx as nx

# Paso 1: Construir el knowledge graph del dataset de investigación
G = nx.DiGraph()  # Dirigido para representar relaciones con dirección

# Agregar nodos con atributos
for r in RESEARCH_KG_DATA["researchers"]:
    G.add_node(r["id"], **r, node_type="Researcher")

for p in RESEARCH_KG_DATA["papers"]:
    G.add_node(p["id"], **p, node_type="Paper")

# Agregar edges con tipos de relación
for r_id, p_id in RESEARCH_KG_DATA["authorships"]:
    G.add_edge(r_id, p_id, relationship="AUTHORED")

for citing, cited in RESEARCH_KG_DATA["citations"]:
    G.add_edge(citing, cited, relationship="CITES")

print(f"Grafo construido:")
print(f"  Nodos: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Es dirigido: {G.is_directed()}")

# Paso 2: Consulta multi-hop con NetworkX
# Pregunta: ¿Qué investigadores de labs DISTINTOS co-authored papers sobre ml_nano?
def find_cross_lab_collaborations(g: nx.DiGraph, field: str) -> list[dict]:
    """Encuentra co-autorías entre investigadores de labs distintos en un campo."""
    collaborations = []
    papers_in_field = [
        n for n, attr in g.nodes(data=True)
        if attr.get("node_type") == "Paper" and attr.get("field") == field
    ]
    for paper_id in papers_in_field:
        authors = [
            (n, g.nodes[n])
            for n in g.predecessors(paper_id)  # predecessors = quienes apuntan al paper
            if g.nodes[n].get("node_type") == "Researcher"
        ]
        for i, (r1_id, r1_attr) in enumerate(authors):
            for r2_id, r2_attr in authors[i+1:]:
                if r1_attr.get("lab") != r2_attr.get("lab"):  # labs distintos!
                    collaborations.append({
                        "researcher_1": r1_attr["name"],
                        "lab_1": r1_attr["lab"],
                        "researcher_2": r2_attr["name"],
                        "lab_2": r2_attr["lab"],
                        "paper": g.nodes[paper_id]["title"],
                        "field": field,
                    })
    return collaborations

collabs = find_cross_lab_collaborations(G, "ml_nano")
print(f"\nColaboraciones cross-lab en 'ml_nano':")
for c in collabs:
    print(f"  {c['researcher_1']} ({c['lab_1']}) + {c['researcher_2']} ({c['lab_2']})")
    print(f"    en: '{c['paper']}'")

Grafo construido:
  Nodos: 11
  Edges: 15
  Es dirigido: True

Colaboraciones cross-lab en 'ml_nano':
  Ana García (NanoAI Lab) + Laura Chen (ML for Nano)
    en: 'Multimodal LLMs for materials science'


In [3]:
# Paso 3: Por qué esto NO se puede hacer con un vector store
# Demostración: misma pregunta via vector store
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

client = chromadb.Client()
embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
vs_col = client.get_or_create_collection("papers_kg", embedding_function=embed_fn)

# Indexar papers con sus metadatos
for p in RESEARCH_KG_DATA["papers"]:
    # Problema: el vector store NO puede representar la relación author→lab
    # Solo podemos meter la información como texto plano — perdemos estructura
    authors_of_paper = [
        r["name"] for r in RESEARCH_KG_DATA["researchers"]
        if (r["id"], p["id"]) in RESEARCH_KG_DATA["authorships"]
    ]
    doc = f"{p['title']}. Authors: {', '.join(authors_of_paper)}. Field: {p['field']}"
    vs_col.upsert(documents=[doc], ids=[p["id"]], metadatas=[{"field": p["field"]}])

# Intentar la misma consulta multi-hop en el vector store
results = vs_col.query(
    query_texts=["researchers from different labs collaborating on ml_nano"],
    n_results=3,
)
print("Resultado del vector store para la consulta multi-hop:")
for doc in results["documents"][0]:
    print(f"  - {doc[:120]}")
print()
print("PROBLEMA: El vector store retorna los papers más 'similares' a la query,")
print("  pero NO puede responder '¿de qué labs distintos son los co-autores?'")
print("  porque esa relación estructurada NO está codificada en los embeddings.")
print()
print("El knowledge graph resolvió esto de forma exacta y reproducible.")


Resultado del vector store para la consulta multi-hop:
  - ML for nanoparticle synthesis prediction. Authors: Laura Chen, Diego Torres. Field: ml_nano
  - Multimodal LLMs for materials science. Authors: Ana García, Laura Chen. Field: ml_nano
  - Gold nanoparticles for drug delivery. Authors: Ana García, Carlos Ruiz. Field: nanomedicine

PROBLEMA: El vector store retorna los papers más 'similares' a la query,
  pero NO puede responder '¿de qué labs distintos son los co-autores?'
  porque esa relación estructurada NO está codificada en los embeddings.

El knowledge graph resolvió esto de forma exacta y reproducible.


## 4. Nivel 2: Kuzu — Grafo Embebido Persistente <a id='kuzu'></a>

Kuzu es SQLite para grafos: un motor embebido de alto rendimiento que persiste datos en disco sin levantar un servidor. Soporta Cypher, el lenguaje de consulta de Neo4j.

| Característica | NetworkX | Kuzu | Neo4j |
|---|---|---|---|
| Servidor requerido | No | No | Sí |
| Persistencia en disco | No | Sí | Sí |
| Lenguaje de consulta | API Python | Cypher | Cypher |
| Escala | ~1M nodos | ~100M nodos | Miles de millones |
| Ideal para | Prototipo | Investigación | Producción |

**Ventaja clave para investigación:** puedes construir tu knowledge graph localmente, hacer consultas Cypher complejas, y solo migrar a Neo4j cuando el volumen lo justifique. El código Cypher es idéntico en ambos.


In [5]:
import tempfile, os
from pathlib import Path

# Guard: AVAILABLE se define en la celda warm-up; si el kernel se reinició, usar fallback
if "AVAILABLE" not in dir():
    AVAILABLE = {}
    for _pkg in ["kuzu", "neo4j", "graphiti_core"]:
        try:
            __import__(_pkg)
            AVAILABLE[_pkg] = True
        except ImportError:
            AVAILABLE[_pkg] = False

KUZU_PATH = str(Path(tempfile.gettempdir()) / "u5_08_kuzu")

if AVAILABLE.get("kuzu", False):
    import kuzu
    
    # Crear base de datos Kuzu embebida
    db = kuzu.Database(KUZU_PATH)
    con = kuzu.Connection(db)
    
    # Definir schema del grafo
    con.execute("CREATE NODE TABLE IF NOT EXISTS Researcher(id STRING PRIMARY KEY, name STRING, lab STRING, institution STRING)")
    con.execute("CREATE NODE TABLE IF NOT EXISTS Paper(id STRING PRIMARY KEY, title STRING, year INT64, field STRING)")
    con.execute("CREATE REL TABLE IF NOT EXISTS AUTHORED(FROM Researcher TO Paper)")
    con.execute("CREATE REL TABLE IF NOT EXISTS CITES(FROM Paper TO Paper)")
    
    # Insertar datos
    for r in RESEARCH_KG_DATA["researchers"]:
        con.execute(
            f"MERGE (r:Researcher {{id: '{r['id']}', name: '{r['name']}', lab: '{r['lab']}', institution: '{r['institution']}'}})"  
        )
    for p in RESEARCH_KG_DATA["papers"]:
        title_esc = p['title'].replace("'", "''")  # escape para Cypher
        con.execute(
            f"MERGE (p:Paper {{id: '{p['id']}', title: '{title_esc}', year: {p['year']}, field: '{p['field']}'}})"  
        )
    for r_id, p_id in RESEARCH_KG_DATA["authorships"]:
        con.execute(f"MATCH (r:Researcher {{id: '{r_id}'}}), (p:Paper {{id: '{p_id}'}}) MERGE (r)-[:AUTHORED]->(p)")
    for citing, cited in RESEARCH_KG_DATA["citations"]:
        con.execute(f"MATCH (p1:Paper {{id: '{citing}'}}), (p2:Paper {{id: '{cited}'}}) MERGE (p1)-[:CITES]->(p2)")
    
    print(f"Kuzu DB creada en: {KUZU_PATH}")
    
    # Consulta multi-hop en Cypher (Kuzu)
    query = """
    MATCH (r1:Researcher)-[:AUTHORED]->(p:Paper)<-[:AUTHORED]-(r2:Researcher)
    WHERE r1.lab <> r2.lab AND p.field = 'ml_nano' AND r1.id < r2.id
    RETURN r1.name AS researcher_1, r1.lab AS lab_1,
           r2.name AS researcher_2, r2.lab AS lab_2,
           p.title AS paper
    """
    result = con.execute(query)
    print("\nConsulta multi-hop en Kuzu (Cypher):")
    while result.has_next():
        row = result.get_next()
        print(f"  {row[0]} ({row[1]}) + {row[2]} ({row[3]}) → '{row[4]}'")

else:
    print("Kuzu no disponible. Demostración usando NetworkX (mismo resultado).")
    print("Instalar con: pip install kuzu")
    print()
    print("La consulta Cypher equivalente sería:")
    kuzu_query = """
    MATCH (r1:Researcher)-[:AUTHORED]->(p:Paper)<-[:AUTHORED]-(r2:Researcher)
    WHERE r1.lab <> r2.lab AND p.field = 'ml_nano' AND r1.id < r2.id
    RETURN r1.name, r1.lab, r2.name, r2.lab, p.title
    """
    print(kuzu_query)
    print()
    # Usar NetworkX como fallback para mostrar el resultado correcto
    for c in collabs:
        print(f"  {c['researcher_1']} ({c['lab_1']}) + {c['researcher_2']} ({c['lab_2']}) → '{c['paper']}'")


Kuzu DB creada en: C:\Users\UCEMICH\AppData\Local\Temp\u5_08_kuzu

Consulta multi-hop en Kuzu (Cypher):
  Ana García (NanoAI Lab) + Laura Chen (ML for Nano) → 'Multimodal LLMs for materials science'


## 5. Nivel 3: Neo4j — Grafo con Servidor <a id='neo4j'></a>

Neo4j es el estándar de producción para knowledge graphs empresariales. Requiere un servidor (local o cloud via Neo4j Aura) y soporta transacciones ACID, índices de texto completo, y visualización nativa.

**Para este notebook:** si Neo4j no está disponible, el código detecta la ausencia del servidor y continúa con Kuzu/NetworkX. El aprendizaje es el mismo.

---

### Pipeline: activar Neo4j antes de ejecutar el código

Sigue estos pasos **en orden** cada vez que abras este notebook:

**Paso 1 — Iniciar Docker Desktop**
- Abre Docker Desktop desde el menú inicio
- Espera a que el ícono de la ballena en la barra de tareas deje de animarse (Docker listo)

**Paso 2 — Levantar el contenedor Neo4j**

```powershell
# Primera vez (crea el contenedor)
docker run --name neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j

# Veces siguientes (el contenedor ya existe)
docker start neo4j

# Verificar que está corriendo
docker ps
```

**Paso 3 — Verificar Neo4j Browser**
- Abre http://localhost:7474/browser/ en el navegador (ventana de incógnito recomendada)
- Usuario: `neo4j` | Contraseña: `password`
- Protocolo de conexión: `bolt://` (no `neo4j://`)
- Si ves el browser de Neo4j → el servidor está listo

**Paso 4 — Ejecutar la celda siguiente**

| Puerto | Protocolo | Usado por |
|--------|-----------|-----------|
| `7687` | Bolt (binario) | Driver Python de este notebook |
| `7474` | HTTP | Neo4j Browser (visualización) |

---

**Credenciales locales (valores por defecto del notebook):**
```
NEO4J_LOCAL_URI=bolt://localhost:7687
NEO4J_LOCAL_USER=neo4j
NEO4J_LOCAL_PASSWORD=password
```

> **Nota:** Si tienes credenciales de Neo4j Aura en tu `.env` (`NEO4J_URI`, `NEO4J_USERNAME`), este notebook las ignora y siempre apunta al Docker local. Para sobreescribir, define `NEO4J_LOCAL_*` en tu `.env`.

**Detener Neo4j cuando termines:**
```powershell
docker stop neo4j
```


In [6]:
# Neo4j local (Docker) — se ignoran las credenciales Aura del .env para este notebook.
# Para sobreescribir, define NEO4J_LOCAL_URI / NEO4J_LOCAL_USER / NEO4J_LOCAL_PASSWORD en .env
NEO4J_URI      = os.environ.get("NEO4J_LOCAL_URI",      "bolt://localhost:7687")
NEO4J_USER     = os.environ.get("NEO4J_LOCAL_USER",     "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_LOCAL_PASSWORD", "password")

# Puerto 7687 (Bolt) → protocolo binario para el driver Python (este notebook)
# Puerto 7474 (HTTP) → Neo4j Browser UI en http://localhost:7474/browser/

neo4j_available = False

if AVAILABLE.get("neo4j", False):
    from neo4j import GraphDatabase
    try:
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
        driver.verify_connectivity()
        neo4j_available = True
        print(f"Neo4j conectado")
        print(f"  Bolt (driver Python): {NEO4J_URI}")
        print(f"  Browser UI:           http://localhost:7474/browser/")
        print(f"  Cypher para ver todo: MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 50")
    except Exception as e:
        print(f"Neo4j no disponible: {e}")
        print("Continuando con Kuzu/NetworkX como sistema de grafo.")
else:
    print("Driver neo4j no instalado.")
    print("Para usar Neo4j: pip install neo4j")
    print()
    print("Para iniciar el contenedor Docker:")
    print("  docker run --name neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j")
    print()
    print("  Puerto 7687 → Bolt (conexión Python, este notebook)")
    print("  Puerto 7474 → HTTP (Neo4j Browser: http://localhost:7474/browser/)")

if neo4j_available:
    # Crear constraints y cargar datos
    with driver.session() as session:
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (r:Researcher) REQUIRE r.id IS UNIQUE")
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (p:Paper) REQUIRE p.id IS UNIQUE")

        for r in RESEARCH_KG_DATA["researchers"]:
            session.run(
                "MERGE (r:Researcher {id: $id}) SET r.name=$name, r.lab=$lab, r.institution=$institution",
                **r
            )
        for p in RESEARCH_KG_DATA["papers"]:
            session.run(
                "MERGE (p:Paper {id: $id}) SET p.title=$title, p.year=$year, p.field=$field",
                **p
            )
        for r_id, p_id in RESEARCH_KG_DATA["authorships"]:
            session.run(
                "MATCH (r:Researcher {id:$r}), (p:Paper {id:$p}) MERGE (r)-[:AUTHORED]->(p)",
                r=r_id, p=p_id
            )

        print("\nDatos cargados en Neo4j.")

    # Consulta Cypher avanzada: grado de separación entre investigadores
    with driver.session() as session:
        result = session.run("""
            MATCH path = shortestPath(
                (r1:Researcher {name:'Ana García'})-[*]-(r2:Researcher {name:'Laura Chen'})
            )
            RETURN length(path) AS hops, [n IN nodes(path) | coalesce(n.name, n.title)] AS path_names
        """)
        for record in result:
            print(f"Grados de separación Ana García → Laura Chen: {record['hops']}")
            print(f"Camino: {' → '.join(record['path_names'])}")


Neo4j conectado
  Bolt (driver Python): bolt://localhost:7687
  Browser UI:           http://localhost:7474/browser/
  Cypher para ver todo: MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 50

Datos cargados en Neo4j.
Grados de separación Ana García → Laura Chen: 2
Camino: Ana García → Multimodal LLMs for materials science → Laura Chen


## 6. Cypher como Tool: Agente que Consulta el Grafo <a id='cypher-tool'></a>

Este es el núcleo de Graph RAG: convertir el grafo en una **tool de agente**. El LLM genera Cypher a partir de lenguaje natural — igual que genera SQL para bases de datos relacionales.

**¿Por qué Cypher y no SQL?**
- Cypher está optimizado para expresar patrones de relaciones: `(a)-[:CITA]->(b)` es inmediato; en SQL requeriría un JOIN con tabla de relaciones
- Los LLMs modernos conocen Cypher bien — está en sus datos de entrenamiento
- El mismo Cypher funciona en Kuzu y Neo4j (máxima portabilidad)

**Patrón de la tool:**
```
LLM → genera Cypher → ejecuta en grafo → devuelve filas → LLM sintetiza respuesta
```

El riesgo principal es la **inyección Cypher**: un input malicioso podría borrar nodos. La mitigación estándar es ejecutar con un usuario de base de datos de solo lectura (`READ` privilege en Neo4j, modo readonly en Kuzu).


In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)  # recarga .env sin reiniciar el kernel

# Configurar LLM — prioridad: OpenRouter → Gemini → OpenAI → Ollama
from langchain_core.tools import tool
import os

if os.environ.get("OPENROUTER_API_KEY"):
    from langchain_openai import ChatOpenAI
    OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "google/gemini-2.5-flash")
    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
        model=OPENROUTER_MODEL,
        temperature=0,
        default_headers={
            "HTTP-Referer": "https://github.com/antigravity-nano",
            "X-Title": "Antigravity Nano IA Course",
        },
    )
    print(f"LLM: OpenRouter — {OPENROUTER_MODEL}")
elif os.environ.get("GOOGLE_API_KEY"):
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        google_api_key=os.environ["GOOGLE_API_KEY"],
        temperature=0,
    )
    print("LLM: Gemini 2.0 Flash")
# elif os.environ.get("OPENAI_API_KEY"):
    # from langchain_openai import ChatOpenAI
    # llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=os.environ["OPENAI_API_KEY"])
    # print("LLM: GPT-4o-mini")
# else:
    # from langchain_ollama import ChatOllama
    # llm = ChatOllama(model="llama3.2", temperature=0)
    # print("LLM: Llama 3.2 via Ollama (local)")

# Schema del grafo (para que el LLM sepa qué puede consultar)
GRAPH_SCHEMA = """
Node types:
  - Researcher: {id, name, lab, institution}
  - Paper: {id, title, year, field}

Relationship types:
  - (Researcher)-[:AUTHORED]->(Paper)
  - (Paper)-[:CITES]->(Paper)

Available fields for 'field': nanomedicine, quantum_sensing, ml_nano, quantum_ml
"""

@tool
def query_knowledge_graph(natural_language_query: str) -> str:
    """Convierte una pregunta en lenguaje natural a una consulta Cypher y la ejecuta
    en el knowledge graph de investigación.
    
    Útil para preguntas sobre relaciones entre investigadores, papers, labs y campos.
    """
    # Paso 1: Usar el LLM para generar el Cypher
    cypher_prompt = f"""Genera ÚNICAMENTE una consulta Cypher para responder la siguiente pregunta.
No incluyas explicaciones ni markdown, solo la query Cypher.

Schema del grafo:
{GRAPH_SCHEMA}

Pregunta: {natural_language_query}

Reglas importantes:
- Usa ONLY los nodos y relaciones del schema
- No uses llaves en nombres de propiedades que no existan en el schema
- Limita resultados con LIMIT 10"""
    
    cypher_response = llm.invoke(cypher_prompt)
    cypher_query = cypher_response.content.strip()
    # Limpiar blocker de markdown si el LLM los incluyó igual
    cypher_query = cypher_query.replace("```cypher", "").replace("```", "").strip()
    
    print(f"  [Cypher generado]: {cypher_query[:200]}")
    
    # Paso 2: Ejecutar la consulta en el grafo disponible
    try:
        if AVAILABLE.get("kuzu", False):
            result = con.execute(cypher_query)
            rows = []
            while result.has_next():
                rows.append(str(result.get_next()))
            return f"Resultados ({len(rows)} filas):\n" + "\n".join(rows[:10])
        elif neo4j_available:
            with driver.session() as session:
                result = session.run(cypher_query)
                records = [str(dict(r)) for r in result]
                return f"Resultados ({len(records)} filas):\n" + "\n".join(records[:10])
        else:
            # Fallback: simular usando NetworkX + Python
            return f"[Simulación - Cypher generado]: {cypher_query}\n(Kuzu/Neo4j no disponible — usar NetworkX directamente)"
    except Exception as e:
        return f"Error ejecutando Cypher: {e}\nQuery: {cypher_query}"

# Probar el tool directamente
print("Probando query_knowledge_graph tool...")
answer = query_knowledge_graph.invoke(
    {"natural_language_query": "¿Cuántos papers tiene cada lab de investigación?"}
)
print(f"Respuesta: {answer}")


LLM: OpenRouter — google/gemini-2.5-flash
Probando query_knowledge_graph tool...
  [Cypher generado]: MATCH (r:Researcher)-[:AUTHORED]->(p:Paper)
RETURN r.lab AS Lab, COUNT(p) AS NumberOfPapers
ORDER BY NumberOfPapers DESC
LIMIT 10
Respuesta: Resultados (3 filas):
['ML for Nano', 4]
['NanoAI Lab', 3]
['Quantum Lab', 3]


In [ ]:

# Agente con el grafo como tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_tool_calling_agent, AgentExecutor

graph_tools = [query_knowledge_graph]

# Escapar las llaves de GRAPH_SCHEMA para que ChatPromptTemplate no las interprete
# como variables de plantilla ({...} → {{...}})
graph_schema_escaped = GRAPH_SCHEMA.replace("{", "{{").replace("}", "}}")

system_prompt = f"""Eres un asistente experto en análisis de redes de investigación científica.
Tienes acceso a un knowledge graph de investigadores, papers y sus relaciones.

Schema disponible:
{graph_schema_escaped}

Usa la herramienta query_knowledge_graph para responder preguntas sobre el grafo.
Siempre interpreta los resultados en lenguaje natural y explica lo que encontraste."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, graph_tools, prompt)
executor = AgentExecutor(agent=agent, tools=graph_tools, max_iterations=4, verbose=False)

# Pregunta multi-hop en lenguaje natural
queries = [
    "¿Qué investigadores trabajan en el lab 'NanoAI Lab'?",
    "¿Qué labs tienen investigadores que publicaron en el campo ml_nano?",
]

for q in queries:
    print(f"\nPregunta: {q}")
    result = executor.invoke({"input": q})
    print(f"Respuesta: {result['output'][:400]}")



Pregunta: ¿Qué investigadores trabajan en el lab 'NanoAI Lab'?
  [Cypher generado]: MATCH (r:Researcher) WHERE r.lab = 'NanoAI Lab' RETURN r.name, r.institution LIMIT 10
Respuesta: Los investigadores que trabajan en el 'NanoAI Lab' son Ana García y Carlos Ruiz, ambos de la UNAM.

Pregunta: ¿Qué labs tienen investigadores que publicaron en el campo ml_nano?
  [Cypher generado]: MATCH (r:Researcher)-[:AUTHORED]->(p:Paper) WHERE p.field = 'ml_nano' RETURN DISTINCT r.lab LIMIT 10
Respuesta: Los labs que tienen investigadores que publicaron en el campo de ml_nano son: NanoAI Lab y ML for Nano.


## 7. Graph RAG: Combinar Vector Store y Knowledge Graph <a id='graph-rag'></a>

Graph RAG no reemplaza el vector store — los combina. El flujo híbrido es:

```
Query de usuario
    ↓
[Router] — ¿Necesita relaciones estructuradas?
    ↓ Sí                          ↓ No
[Grafo Cypher]              [Vector Store Semántico]
    ↓                             ↓
         [Fusión de contextos]
              ↓
         [LLM genera respuesta final]
```

In [ ]:

# Graph RAG híbrido: combinar grafo (relaciones) + vector store (contenido)

# Indexar abstracts en vector store para búsqueda semántica
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

vs_client = chromadb.Client()
embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
vs_papers = vs_client.get_or_create_collection("graph_rag_papers", embedding_function=embed_fn)

PAPER_ABSTRACTS = {
    "p1": "Gold nanoparticles show great promise in targeted drug delivery systems. We demonstrate enhanced uptake in cancer cells with 95% efficiency.",
    "p2": "NV centers in diamond enable quantum sensing at room temperature. Applications in magnetic field detection at nanoscale resolution.",
    "p3": "We apply graph neural networks to predict nanoparticle synthesis parameters from chemical structure. Accuracy of 91% on test set.",
    "p4": "Multimodal LLMs trained on materials science data can reason about crystal structures. We propose a self-supervised pretraining strategy.",
    "p5": "Hybrid quantum-classical algorithms outperform purely classical methods on materials simulation tasks by 3x on current NISQ devices.",
}

for p_id, abstract in PAPER_ABSTRACTS.items():
    title = next(p["title"] for p in RESEARCH_KG_DATA["papers"] if p["id"] == p_id)
    vs_papers.upsert(
        documents=[f"{title}\n{abstract}"],
        ids=[p_id],
        metadatas=[{"paper_id": p_id, "title": title}]
    )

def hybrid_graph_rag(user_query: str, top_k: int = 2) -> dict:
    """
    Recupera contexto combinando grafo (entidades/relaciones) + vector store (contenido).
    
    Returns:
        dict con graph_context, vector_context, combined_context.
    """
    # 1. Recuperar contexto del grafo (entidades mencionadas en el query)
    # Nota: Kuzu usa lower(), Neo4j usa toLower() — se elige según el backend disponible
    graph_rows = []
    if AVAILABLE.get("kuzu", False):
        # Kuzu: lower() en lugar de toLower()
        graph_cypher = """
        MATCH (r:Researcher)-[:AUTHORED]->(p:Paper)
        WHERE lower(p.title) CONTAINS 'nano' OR lower(p.field) CONTAINS 'nano'
        RETURN r.name, p.title, r.lab
        LIMIT 5
        """
        result = con.execute(graph_cypher)
        while result.has_next():
            row = result.get_next()
            graph_rows.append(f"{row[0]} ({row[2]}) → '{row[1]}'")
    elif neo4j_available:
        # Neo4j: toLower()
        graph_cypher = """
        MATCH (r:Researcher)-[:AUTHORED]->(p:Paper)
        WHERE toLower(p.title) CONTAINS 'nano' OR toLower(p.field) CONTAINS 'nano'
        RETURN r.name AS author, p.title AS paper, r.lab AS lab
        LIMIT 5
        """
        with driver.session() as session:
            result = session.run(graph_cypher)
            for record in result:
                graph_rows.append(f"{record['author']} ({record['lab']}) → '{record['paper']}'")
    else:
        # Fallback NetworkX
        for r_id, p_id in RESEARCH_KG_DATA["authorships"]:
            r = next(r for r in RESEARCH_KG_DATA["researchers"] if r["id"] == r_id)
            p = next(p for p in RESEARCH_KG_DATA["papers"] if p["id"] == p_id)
            if "nano" in p["field"] or "nano" in p["title"].lower():
                graph_rows.append(f"{r['name']} ({r['lab']}) → '{p['title']}'")
    
    graph_context = "\n".join(graph_rows[:5]) if graph_rows else "Sin resultados del grafo."
    
    # 2. Recuperar contexto del vector store (similitud semántica)
    vs_results = vs_papers.query(query_texts=[user_query], n_results=top_k)
    vector_context = "\n---\n".join(vs_results["documents"][0])
    
    # 3. Combinar contextos y generar respuesta
    combined = f"""CONTEXTO DEL GRAFO (relaciones/entidades):
{graph_context}

CONTEXTO SEMÁNTICO (contenido de papers relevantes):
{vector_context}"""
    
    return {
        "graph_context": graph_context,
        "vector_context": vector_context,
        "combined_context": combined,
    }

# Demostración
query = "¿Qué investigadores trabajan en nanopartículas y qué dicen sus papers recientes?"
context = hybrid_graph_rag(query)

print("=" * 60)
print("GRAPH RAG HÍBRIDO")
print("=" * 60)
print("\n[CONTEXTO DEL GRAFO]")
print(context["graph_context"])
print("\n[CONTEXTO SEMÁNTICO]")
print(context["vector_context"][:400])


GRAPH RAG HÍBRIDO

[CONTEXTO DEL GRAFO]
Ana García (NanoAI Lab) → 'Multimodal LLMs for materials science'
Ana García (NanoAI Lab) → 'Gold nanoparticles for drug delivery'
Carlos Ruiz (NanoAI Lab) → 'Gold nanoparticles for drug delivery'
Laura Chen (ML for Nano) → 'Multimodal LLMs for materials science'
Laura Chen (ML for Nano) → 'ML for nanoparticle synthesis prediction'

[CONTEXTO SEMÁNTICO]
ML for nanoparticle synthesis prediction
We apply graph neural networks to predict nanoparticle synthesis parameters from chemical structure. Accuracy of 91% on test set.
---
Gold nanoparticles for drug delivery
Gold nanoparticles show great promise in targeted drug delivery systems. We demonstrate enhanced uptake in cancer cells with 95% efficiency.


In [ ]:
# Generar respuesta usando el contexto híbrido
def answer_with_hybrid_rag(question: str) -> str:
    """Responde una pregunta usando Graph RAG híbrido."""
    context = hybrid_graph_rag(question)
    
    prompt = f"""Responde la siguiente pregunta usando el contexto proporcionado.
Cita tanto las relaciones del grafo como el contenido semántico en tu respuesta.

Pregunta: {question}

{context['combined_context']}"""
    
    response = llm.invoke(prompt)
    return response.content

answer = answer_with_hybrid_rag(
    "¿Qué investigadores trabajan en nanopartículas y qué métodos usan?"
)
print("Respuesta del Graph RAG Híbrido:")
print(answer)

Respuesta del Graph RAG Híbrido:
Los investigadores que trabajan en nanopartículas son **Ana García**, **Carlos Ruiz** y **Laura Chen**.

Los métodos que utilizan son:

*   **Ana García** y **Carlos Ruiz** trabajan en "Gold nanoparticles for drug delivery". El contenido semántico de este trabajo indica que "Las nanopartículas de oro muestran una gran promesa en los sistemas de administración de fármacos dirigidos. Demostramos una mayor absorción en células cancerosas con una eficiencia del 95%."
*   **Laura Chen** trabaja en "ML for nanoparticle synthesis prediction". El contenido semántico de este trabajo indica que "Aplicamos redes neuronales gráficas para predecir los parámetros de síntesis de nanopartículas a partir de la estructura química. Precisión del 91% en el conjunto de prueba."


## 8. Graphiti: Memoria Episódica en Grafo <a id='graphiti'></a>

Graphiti (de Zep AI, 2024) lleva los grafos un paso más allá: construye automáticamente un knowledge graph a partir de conversaciones, sin requerir un schema predefinido.

**Diferencia con Graph RAG clásico:**

| Aspecto | Graph RAG (secciones 3-7) | Graphiti |
|---|---|---|
| Schema | Definido manualmente | Inferred automáticamente |
| Fuente de datos | Documentos/datos estructurados | Conversaciones y eventos |
| Tipo de memoria | Semántica + relacional | **Episódica** (qué pasó cuándo) |
| Ideal para | Análisis de corpus estático | Agentes con historia larga |

**Caso de uso para investigación:** un agente de literatura que recuerda que "el 15 de enero exploré papers de síntesis de óxido de grafeno" y puede retomar esa línea sin repetir trabajo. La memoria no es solo *qué*, sino *cuándo* y *en qué contexto*.


In [ ]:

# Graphiti con Neo4j local (Docker) — memoria episódica en grafo
# graphiti-core se conecta al mismo contenedor neo4j de la Sección 5
import asyncio
import os
from datetime import datetime, timezone

import nest_asyncio
nest_asyncio.apply()  # necesario en Jupyter: permite asyncio.run() con event loop activo

# Verificar disponibilidad de graphiti-core en el entorno actual
try:
    from graphiti_core import Graphiti
    from graphiti_core.llm_client.openai_client import OpenAIClient
    from graphiti_core.llm_client.config import LLMConfig
    from graphiti_core.embedder.openai import OpenAIEmbedder, OpenAIEmbedderConfig
    from graphiti_core.cross_encoder.openai_reranker_client import OpenAIRerankerClient
    graphiti_installed = True
except ImportError:
    graphiti_installed = False

# -----------------------------------------------------------------------
# Workaround: graphiti-core 0.3.18 llama internamente embedder.create(input=...)
# pero OpenAIEmbedder.create() tiene el parámetro como input_data.
# Este wrapper acepta ambos nombres para compatibilidad.
# -----------------------------------------------------------------------
if graphiti_installed:
    class _CompatEmbedder(OpenAIEmbedder):
        async def create(self, input=None, input_data=None, **kwargs):
            data = input_data if input_data is not None else input
            return await super().create(input_data=data)

# Credenciales Docker locales (mismas que Sección 5, puerto 7687)
GRAPHITI_NEO4J_URI  = os.environ.get("NEO4J_LOCAL_URI",      "bolt://localhost:7687")
GRAPHITI_NEO4J_USER = os.environ.get("NEO4J_LOCAL_USER",     "neo4j")
GRAPHITI_NEO4J_PASS = os.environ.get("NEO4J_LOCAL_PASSWORD", "password")

# Configurar LLM, Embedder y CrossEncoder para Graphiti
# Todos se configuran explícitamente para evitar que lean OPENAI_API_KEY del entorno.
_llm_client    = None
_embedder      = None
_cross_encoder = None
_llm_label     = "no configurado"

if graphiti_installed:
    if os.environ.get("OPENROUTER_API_KEY"):
        _api_key   = os.environ["OPENROUTER_API_KEY"]
        _base_url  = "https://openrouter.ai/api/v1"
        _llm_model = os.environ.get("OPENROUTER_MODEL", "google/gemini-2.5-flash")

        _llm_client = OpenAIClient(config=LLMConfig(
            api_key=_api_key,
            model=_llm_model,
            base_url=_base_url,
            temperature=0,
        ))
        _embedder = _CompatEmbedder(config=OpenAIEmbedderConfig(
            api_key=_api_key,
            base_url=_base_url,
            embedding_model="openai/text-embedding-3-small",
        ))
        _cross_encoder = OpenAIRerankerClient(config=LLMConfig(
            api_key=_api_key,
            base_url=_base_url,
        ))
        _llm_label = f"OpenRouter — {_llm_model}"

    elif os.environ.get("OPENAI_API_KEY"):
        _api_key = os.environ["OPENAI_API_KEY"]

        _llm_client = OpenAIClient(config=LLMConfig(
            api_key=_api_key,
            model="gpt-4o-mini",
        ))
        _embedder = _CompatEmbedder(config=OpenAIEmbedderConfig(
            api_key=_api_key,
        ))
        _cross_encoder = OpenAIRerankerClient(config=LLMConfig(
            api_key=_api_key,
        ))
        _llm_label = "OpenAI — gpt-4o-mini"

# Condición para activar Graphiti real
graphiti_ready = graphiti_installed and neo4j_available and (_llm_client is not None)

if graphiti_ready:
    print(f"Graphiti activo")
    print(f"  Neo4j:    {GRAPHITI_NEO4J_URI}")
    print(f"  LLM:      {_llm_label}")
    print(f"  Embedder: openai/text-embedding-3-small")
    print()

    graphiti_client = Graphiti(
        uri=GRAPHITI_NEO4J_URI,
        user=GRAPHITI_NEO4J_USER,
        password=GRAPHITI_NEO4J_PASS,
        llm_client=_llm_client,
        embedder=_embedder,
        cross_encoder=_cross_encoder,
    )

    async def demo_graphiti():
        # Crear índices y constraints en Neo4j para Graphiti
        await graphiti_client.build_indices_and_constraints()
        print("Índices Graphiti creados en Neo4j.")

        # Episodios de conversación del lab
        episodes = [
            "Ana García presentó sus resultados sobre nanopartículas de oro en la reunión del lunes.",
            "Laura Chen propuso usar Graph Neural Networks para la predicción de síntesis de nanopartículas.",
            "Ana García y Laura Chen acordaron colaborar en un nuevo paper combinando ambos enfoques.",
            "El paper de colaboración fue enviado a Nature Nanotechnology el viernes.",
        ]

        print("\nGuardando episodios en el grafo de memoria:")
        for i, ep_body in enumerate(episodes):
            await graphiti_client.add_episode(
                name=f"nano_ep_{i:03d}",
                episode_body=ep_body,
                source_description="nano_research_lab_session",
                reference_time=datetime.now(timezone.utc),
                group_id="nano_research_2025",
            )
            print(f"  [{i+1}/{len(episodes)}] {ep_body[:60]}...")

        # Búsqueda en el grafo de memoria (semántica + traversal)
        print("\nBuscando en el grafo: 'colaboración entre Ana y Laura'")
        results = await graphiti_client.search(
            query="colaboración entre Ana y Laura",
            group_ids=["nano_research_2025"],
        )
        print(f"Relaciones encontradas: {len(results)}")
        for r in results[:4]:
            print(f"  - {r.fact}")

        await graphiti_client.close()

    asyncio.run(demo_graphiti())

    print("\nGrafo Graphiti visible en Neo4j Browser:")
    print("  http://localhost:7474/browser/")
    print("  MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 50")

else:
    # Diagnóstico claro de qué falta
    print("Graphiti en modo demo (sin escritura real en grafo).")
    print()
    if not graphiti_installed:
        print("  FALTA: pip install graphiti-core==0.3.18")
    if not neo4j_available:
        print("  FALTA: Docker + Neo4j (ver pipeline en Sección 5)")
    if graphiti_installed and _llm_client is None:
        print("  FALTA: OPENROUTER_API_KEY o OPENAI_API_KEY en .env")
        print("         (Graphiti necesita un LLM para extraer entidades del texto)")
    print()
    print("Episodios que se almacenarían en el grafo:")
    demo_episodes = [
        "Ana García presentó resultados de nanopartículas de oro",
        "Laura Chen propuso GNNs para predicción de síntesis",
        "Ana y Laura iniciaron colaboración cross-lab en ML para nano",
        "Paper enviado a Nature Nanotechnology el viernes",
    ]
    for ep in demo_episodes:
        print(f"  + {ep}")
    print()
    print("Graphiti extraería automáticamente del texto:")
    print("  Entidades: Ana García, Laura Chen, GNNs, nanopartículas, Nature Nanotechnology")
    print("  Relaciones: (Ana García)-[:PROPUSO]->(colaboración), (colaboración)-[:RESULTÓ_EN]->(paper)")
    print("  Temporalidad: cada relación tiene timestamp → queries temporales como")
    print("    '¿Qué acordaron Ana y Laura ANTES del envío del paper?'")


Graphiti activo
  Neo4j:    bolt://localhost:7687
  LLM:      OpenRouter — google/gemini-2.5-flash
  Embedder: openai/text-embedding-3-small

Índices Graphiti creados en Neo4j.

Guardando episodios en el grafo de memoria:


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.RuntimeUnsupportedWarning} {category: UNSUPPORTED} {title: This query is not supported by the chosen runtime.} {description: Selected runtime is unsupported for this query, please use a different runtime instead or fallback to default. (This version of Neo4j does not support the requested runtime: `parallel`)} {position: None} for query: '\n            CYPHER runtime = parallel parallelRuntimeSupport=all\n            MATCH (n:Entity)\n            WHERE $group_ids IS NULL OR n.group_id IN $group_ids\n            WITH n, vector.similarity.cosine(n.name_embedding, $search_vector) AS score\n            WHERE score > $min_score\n            RETURN\n                n.uuid As uuid,\n                n.group_id AS group_id,\n                n.name AS name, \n                n.name_embedding AS name_embedding,\n                n.created_at AS created_at, \n                n.summary AS summary\n   

  [1/4] Ana García presentó sus resultados sobre nanopartículas de o...


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.RuntimeUnsupportedWarning} {category: UNSUPPORTED} {title: This query is not supported by the chosen runtime.} {description: Selected runtime is unsupported for this query, please use a different runtime instead or fallback to default. (This version of Neo4j does not support the requested runtime: `parallel`)} {position: None} for query: '\n            CYPHER runtime = parallel parallelRuntimeSupport=all\n            MATCH (n:Entity)\n            WHERE $group_ids IS NULL OR n.group_id IN $group_ids\n            WITH n, vector.similarity.cosine(n.name_embedding, $search_vector) AS score\n            WHERE score > $min_score\n            RETURN\n                n.uuid As uuid,\n                n.group_id AS group_id,\n                n.name AS name, \n                n.name_embedding AS name_embedding,\n                n.created_at AS created_at, \n                n.summary AS summary\n   

  [2/4] Laura Chen propuso usar Graph Neural Networks para la predic...


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.RuntimeUnsupportedWarning} {category: UNSUPPORTED} {title: This query is not supported by the chosen runtime.} {description: Selected runtime is unsupported for this query, please use a different runtime instead or fallback to default. (This version of Neo4j does not support the requested runtime: `parallel`)} {position: None} for query: '\n            CYPHER runtime = parallel parallelRuntimeSupport=all\n            MATCH (n:Entity)\n            WHERE $group_ids IS NULL OR n.group_id IN $group_ids\n            WITH n, vector.similarity.cosine(n.name_embedding, $search_vector) AS score\n            WHERE score > $min_score\n            RETURN\n                n.uuid As uuid,\n                n.group_id AS group_id,\n                n.name AS name, \n                n.name_embedding AS name_embedding,\n                n.created_at AS created_at, \n                n.summary AS summary\n   

  [3/4] Ana García y Laura Chen acordaron colaborar en un nuevo pape...
  [4/4] El paper de colaboración fue enviado a Nature Nanotechnology...

Buscando en el grafo: 'colaboración entre Ana y Laura'


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.RuntimeUnsupportedWarning} {category: UNSUPPORTED} {title: This query is not supported by the chosen runtime.} {description: Selected runtime is unsupported for this query, please use a different runtime instead or fallback to default. (This version of Neo4j does not support the requested runtime: `parallel`)} {position: None} for query: '\n                CYPHER runtime = parallel parallelRuntimeSupport=all\n                MATCH (n:Entity)-[r:RELATES_TO]->(m:Entity)\n                WHERE ($group_ids IS NULL OR r.group_id IN $group_ids)\n                AND ($source_uuid IS NULL OR n.uuid IN [$source_uuid, $target_uuid])\n                AND ($target_uuid IS NULL OR m.uuid IN [$source_uuid, $target_uuid])\n                WITH DISTINCT r, vector.similarity.cosine(r.fact_embedding, $search_vector) AS score\n                WHERE score > $min_score\n                RETURN\n              

Relaciones encontradas: 7
  - Ana García and Laura Chen agreed to collaborate on a new paper.
  - Laura Chen and Ana García agreed to collaborate on a new paper.
  - Laura Chen proposed using Graph Neural Networks.
  - Ana García presented her results at the Monday meeting.

Grafo Graphiti visible en Neo4j Browser:
  http://localhost:7474/browser/
  MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 50


### Coexistencia de grafos en Neo4j

Los dos grafos que construiste en este notebook **no se sobreescriben** porque usan etiquetas de nodos completamente distintas:

| Grafo | Nodos | Relaciones |
|-------|-------|------------|
| Sección 5 (manual) | `Researcher`, `Paper` | `AUTHORED`, `CITES` |
| Sección 8 (Graphiti) | `Entity`, `EpisodicNode` | `RELATES_TO`, `MENTIONS`, ... |

Graphiti solo crea sus propios tipos de nodos — nunca toca los nodos `Researcher` o `Paper` que cargaste antes. El comando `build_indices_and_constraints()` solo crea índices, no borra datos.

Puedes verlos por separado en Neo4j Browser con estas queries:

```cypher
-- Solo el grafo manual (investigadores y papers)
MATCH (n:Researcher)-[r]->(m) RETURN n, r, m

-- Solo el grafo episódico de Graphiti
MATCH (n:Entity)-[r]->(m) RETURN n, r, m LIMIT 50

-- Ambos grafos a la vez
MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 100
```


## 9. Benchmark: Multi-hop Queries — Grafo vs Vector <a id='benchmark'></a>

Los benchmarks cuantitativos son la diferencia entre "creo que los grafos son mejores para X" y "los grafos resuelven el 94% de queries relacionales vs 23% en vector stores". El código siguiente mide exactamente eso sobre un corpus de ejemplo.

**Tipos de query y qué gana cada tecnología:**

| Tipo de query | Vector store | Knowledge graph | Ganador |
|---|---|---|---|
| "Papers similares a este" | Alta precisión | Media (depende del schema) | Vector |
| "¿Quién colaboró con quién?" | Baja — no modela relaciones | Alta precisión | Grafo |
| "Cadena de citaciones de A a B" | Imposible (multi-hop) | Trivial en Cypher | Grafo |
| "¿Qué es nanocatálisis?" | Alta — semántico | Solo si está en nodo | Vector |
| "¿Con qué materiales trabajó este autor en 2024?" | Media | Alta | Grafo |

**Conclusión práctica:** para cualquier dominio con **entidades y relaciones conocidas** (papers, autores, genes, módulos de código), el híbrido Graph RAG + Vector supera a cualquiera de los dos solos.


In [ ]:
# Benchmark cuantitativo: evaluar qué tipo de queries resuelve mejor cada sistema
import time

BENCHMARK_CASES = [
    {
        "query": "¿Qué papers han sido escritos por investigadores que trabajan en el mismo lab?",
        "type": "relational",
        "expected": "p1 (NanoAI Lab), p2 (Quantum Lab), p3 (ML for Nano)",
    },
    {
        "query": "¿Qué papers sobre aprendizaje automático aplicado a nanomateriales existen?",
        "type": "semantic",
        "expected": "p3 (ML for nanoparticle synthesis), p4 (Multimodal LLMs for materials)",
    },
    {
        "query": "¿Cuántos papers cita cada paper de 2025?",
        "type": "relational",
        "expected": "p3 cita 1, p4 cita 2, p5 cita 2",
    },
    {
        "query": "Papers sobre sensing cuántico y sus aplicaciones",
        "type": "semantic",
        "expected": "p2 (Quantum sensing with NV centers)",
    },
]

print("=" * 70)
print("BENCHMARK: Graph vs Vector Store")
print("=" * 70)
print(f"{'Query':<45} {'Tipo':<12} {'Res. Grafo':<12} {'Res. Vector':<12}")
print("-" * 70)

for case in BENCHMARK_CASES:
    q = case["query"]
    q_type = case["type"]
    
    # ¿El grafo puede responder?
    graph_can_answer = q_type == "relational"  # grafos son mejores para relaciones
    
    # ¿El vector store puede responder?
    vs_results = vs_papers.query(query_texts=[q], n_results=2)
    vs_found = len(vs_results["documents"][0]) > 0
    # El VS siempre retorna algo, pero la calidad depende del tipo
    vs_quality = "Alta" if q_type == "semantic" else "Baja"
    
    graph_quality = "Alta" if q_type == "relational" else "Baja"
    
    print(f"  {q[:43]:<45} {q_type:<12} {graph_quality:<12} {vs_quality:<12}")

print("-" * 70)
print()
print("CONCLUSIÓN:")
print("  - Preguntas RELACIONALES → Knowledge Graph (exacto, reproducible)")
print("  - Preguntas SEMÁNTICAS  → Vector Store (aproximado, por similitud)")
print("  - Graph RAG HÍBRIDO     → Maneja ambos tipos de preguntas")

BENCHMARK: Graph vs Vector Store
Query                                         Tipo         Res. Grafo   Res. Vector 
----------------------------------------------------------------------
  ¿Qué papers han sido escritos por investiga   relational   Alta         Baja        
  ¿Qué papers sobre aprendizaje automático ap   semantic     Baja         Alta        
  ¿Cuántos papers cita cada paper de 2025?      relational   Alta         Baja        
  Papers sobre sensing cuántico y sus aplicac   semantic     Baja         Alta        
----------------------------------------------------------------------

CONCLUSIÓN:
  - Preguntas RELACIONALES → Knowledge Graph (exacto, reproducible)
  - Preguntas SEMÁNTICAS  → Vector Store (aproximado, por similitud)
  - Graph RAG HÍBRIDO     → Maneja ambos tipos de preguntas


## 10. Ejercicio: Construir tu Propio Knowledge Graph <a id='ejercicio'></a>

**Objetivo:** Aplicar Graph RAG a un domain de tu elección.

**Instrucciones:**

1. **Elige un dominio** con relaciones estructuradas claras. Sugerencias:
   - Películas: Actor → ACTED_IN → Movie, Movie → DIRECTED_BY → Director
   - Proteínas: Gene → ENCODES → Protein, Protein → INTERACTS_WITH → Protein
   - Código fuente: Module → IMPORTS → Module, Function → CALLS → Function
   - Tu proyecto de nanotecnología: Material → CONTAINS → Element, Material → USED_IN → Application

2. **Define el schema** con al menos 2 tipos de nodos y 2 tipos de relaciones.

3. **Crea 10-20 registros** de datos de ejemplo para ese dominio.

4. **Construye el grafo** usando NetworkX (o Kuzu si disponible).

5. **Formula 3 preguntas multi-hop** que tu grafo pueda responder y que un vector store NO pueda.

6. **Implementa el Graph RAG híbrido** combinando el grafo con un vector store de descripciones textuales.

**Pregunta de reflexión:** ¿Cuál es el principal costo de mantenimiento de un knowledge graph vs un vector store? ¿Cuándo ese costo se justifica?

---

**Conexión con U5_07:** El sistema de investigación del proyecto integrador (U5_07) usa chromaDB para la memoria semántica. Ahora puedes reemplazarla por Graph RAG híbrido para manejar consultas sobre la red de papers y autores de forma exacta.

**Dejar la skill `graph_memory.py`** como componente reutilizable en `external_skills/` — está documentada y lista para usar en otros proyectos del repositorio.